# ML Pipeline DAG

This notebook creates an automated ML retraining pipeline using Snowflake Tasks. The pipeline implements the **Champion/Challenger pattern** - a production ML best practice where new models must prove they're better than the current production model before being promoted.

## Pipeline Overview

```
┌──────────────────────────┐        ┌──────────────────────────┐
│ TASK_TRAIN_CHALLENGER    │◀───────│ TASK_CHECK_DRIFT         │
│ (Root - Daily 2AM PST)   │ EXECUTE│ (Daily 2PM PST)          │
│ Train new "challenger"   │  TASK  │ Model Monitor PSI > 0.2  │
│ model and register it    │        │ on any feature -> retrain│
└───────────┬──────────────┘        └──────────────────────────┘
            ▼
┌──────────────────────────┐
│ TASK_VALIDATE_CHALLENGER │  (Runs after training)
│ Compare AUC-PR and       │
│ recall@1%FPR vs champion │
└───────────┬──────────────┘
            ▼
┌──────────────────────────┐
│ TASK_PROMOTE_IF_BETTER   │  (Runs after validation)
│ Promote if AUC-PR or     │
│ recall@1%FPR improves >1%│
│ and neither drops >1%    │
└──────────────────────────┘
```

## Prerequisites

This pipeline reuses artifacts created in earlier notebooks:
- `FRAUD_TRAINING_DATA` table (from `02_feature_engineering.ipynb`)
- `FRAUD_DETECTION_MODEL` in Model Registry (from `03a_model_train.ipynb`)
- `FRAUD_DETECTION_MONITOR` Model Monitor (from `05_model_monitoring.ipynb`) for drift-triggered retraining

Procedure logic lives in [`pipeline/dag_procs.py`](dag_procs.py) (unit-tested in `tests/`) and is registered here with `session.sproc.register_from_file`.

In [ ]:
import os
import sys
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import FloatType, IntegerType, StringType
from snowflake.ml.registry import Registry
from datetime import datetime

session = get_active_session()
session.sql("USE ROLE ACCOUNTADMIN").collect()

DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
WAREHOUSE = "ML_WH"
COMPUTE_POOL = "ML_TRAINING_POOL"
MODEL_NAME = "FRAUD_DETECTION_MODEL"
MONITOR_NAME = f"{DATABASE}.{SCHEMA}.FRAUD_DETECTION_MONITOR"
PSI_DRIFT_THRESHOLD = 0.2
DRIFT_LOOKBACK_DAYS = 1
SOURCE_STAGE = f"@{DATABASE}.{SCHEMA}.ML_MODELS/source"
PROC_STAGE = f"@{DATABASE}.{SCHEMA}.ML_MODELS/procs"

PIPELINE_DIR = os.path.abspath('.') if os.path.exists('dag_procs.py') else os.path.abspath('pipeline')
SOURCE_DIR = os.path.join(os.path.dirname(PIPELINE_DIR), 'source')
sys.path.extend([PIPELINE_DIR, SOURCE_DIR])
import dag_procs

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
versions_df = model_ref.show_versions()
latest = versions_df.sort_values('created_on', ascending=False).iloc[0]
LATEST_VERSION = latest['name']

metrics = model_ref.version(LATEST_VERSION).show_metrics()
metrics = metrics if isinstance(metrics, dict) else {}
LATEST_AUC = float(metrics.get('auc_roc', metrics.get('auc', 0.0)) or 0.0)
LATEST_AUC_PR = float(metrics.get('auc_pr', 0.0) or 0.0)
LATEST_RECALL_1PCT = float(metrics.get('recall_at_1pct_fpr', 0.0) or 0.0)

print(f"Context: {DATABASE}.{SCHEMA}")
print(f"Current best model: {LATEST_VERSION} (AUC-PR={LATEST_AUC_PR:.4f}, Recall@1%FPR={LATEST_RECALL_1PCT:.4f}, AUC-ROC={LATEST_AUC:.4f})")
print(f"Compute pool: {COMPUTE_POOL}")

## 1. Setup Tracking Tables

Create tables to track pipeline runs, the current champion, and drift checks (`dag_procs.ensure_tables` also adds columns introduced by newer versions to existing tables):

- **PIPELINE_RUN_LOG**: Records each step of every pipeline run (train, validate, promote, drift check) with AUC-PR, recall@1%FPR, AUC-ROC and decision thresholds
- **MODEL_CHAMPION_TRACKER**: Current champion version, its AUC-PR / recall@1%FPR / AUC-ROC and decision thresholds
- **MODEL_DRIFT_LOG**: Per-feature PSI from each drift check

Metrics of versions registered before AUC-PR was tracked are 0, so the first challenger trained with the new code is promoted against them.

In [ ]:
dag_procs.ensure_tables(session, DATABASE, SCHEMA)

session.sql(f"""
    MERGE INTO {DATABASE}.{SCHEMA}.MODEL_CHAMPION_TRACKER t
    USING (SELECT ? AS MODEL_NAME, ? AS VER, ?::FLOAT AS AUC, ?::FLOAT AS AUC_PR, ?::FLOAT AS RECALL_1PCT) s
    ON t.MODEL_NAME = s.MODEL_NAME
    WHEN MATCHED THEN UPDATE SET CHAMPION_VERSION = s.VER, CHAMPION_AUC = s.AUC, CHAMPION_AUC_PR = s.AUC_PR,
        CHAMPION_RECALL_AT_1PCT_FPR = s.RECALL_1PCT, UPDATED_AT = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN INSERT (MODEL_NAME, CHAMPION_VERSION, CHAMPION_AUC, CHAMPION_AUC_PR, CHAMPION_RECALL_AT_1PCT_FPR, UPDATED_AT)
        VALUES (s.MODEL_NAME, s.VER, s.AUC, s.AUC_PR, s.RECALL_1PCT, CURRENT_TIMESTAMP())
""", params=[MODEL_NAME, LATEST_VERSION, LATEST_AUC, LATEST_AUC_PR, LATEST_RECALL_1PCT]).collect()

champ = dag_procs.get_champion(session, DATABASE, SCHEMA)
print(f"Champion: {champ['CHAMPION_VERSION']} (AUC-PR={champ['CHAMPION_AUC_PR']}, Recall@1%FPR={champ['CHAMPION_RECALL_AT_1PCT_FPR']})")

## 2. Upload Training Code & Register Challenger Procedure

Instead of duplicating training logic, the `TRAIN_CHALLENGER_MODEL` procedure (`dag_procs.train_challenger`) reuses `source/train.py` (the same script used by notebook 03a) by:
1. Uploading `source/` (`train.py` + shared `utils.py`) to a stage once
2. The proc downloads it to `/tmp`, then calls `submit_directory()` to run as an **ML Job on SPCS**
3. Passes `VERSION_NAME`, `REGISTER_MODEL=true` and the current `CHAMPION_VERSION` so `train.py` registers the challenger and also scores the champion **on the same time-based holdout**
4. Polls until the job completes, then reads `runs/<version>/metrics.pkl` from the stage

This eliminates code duplication — `source/train.py` is the **single source of truth** for training logic.

In [ ]:
session.sql(f"CREATE STAGE IF NOT EXISTS {DATABASE}.{SCHEMA}.ML_MODELS").collect()

for f in os.listdir(SOURCE_DIR):
    if f.endswith('.py'):
        session.file.put(os.path.join(SOURCE_DIR, f), SOURCE_STAGE, auto_compress=False, overwrite=True)
        print(f"Uploaded {f} to {SOURCE_STAGE}")
print(f"\nSource code staged at {SOURCE_STAGE}")

PROCS_FILE = os.path.join(PIPELINE_DIR, "dag_procs.py")
UTILS_FILE = os.path.join(SOURCE_DIR, "utils.py")
BASE_PACKAGES = ["snowflake-snowpark-python", "numpy", "pandas"]

def register_proc(func_name, proc_name, input_types, packages=(), execute_as="owner"):
    session.sproc.register_from_file(
        file_path=PROCS_FILE,
        func_name=func_name,
        name=f"{DATABASE}.{SCHEMA}.{proc_name}",
        return_type=StringType(),
        input_types=input_types,
        is_permanent=True,
        stage_location=PROC_STAGE,
        imports=[UTILS_FILE],
        packages=BASE_PACKAGES + list(packages),
        execute_as=execute_as,
        replace=True,
    )
    print(f"Registered {proc_name} -> dag_procs.{func_name}")

register_proc(
    "train_challenger", "TRAIN_CHALLENGER_MODEL", [StringType()] * 3,
    packages=["snowflake-ml-python", "xgboost", "scikit-learn"], execute_as="caller",
)

## 3. Validate & Promote Procedures

### VALIDATE_CHALLENGER (`dag_procs.validate_challenger` → `decide_promotion`)
Compares the newly trained challenger against the current champion on **AUC-PR** and **recall at 1% FPR** (AUC-ROC is logged for reference only — with ~2% fraud it saturates and barely moves):
- Baseline: the champion's metrics **on the challenger's holdout** when `train.py` could score it (`COMPARISON_BASIS = same_holdout`), otherwise the stored `MODEL_CHAMPION_TRACKER` values
- Decision: `PROMOTE` if AUC-PR **or** recall@1%FPR improves by >1% (relative) **and** neither regresses by >1%; else `KEEP_CHAMPION`

### PROMOTE_IF_BETTER (`dag_procs.promote_if_better`)
Conditionally promotes the challenger to become the new champion:
- If validation decided `PROMOTE`: Updates `MODEL_CHAMPION_TRACKER` (metrics + decision thresholds)
- If validation decided `KEEP_CHAMPION`: Logs skip and retains current champion

In [ ]:
register_proc("validate_challenger", "VALIDATE_CHALLENGER", [StringType()] * 2)

In [ ]:
register_proc("promote_if_better", "PROMOTE_IF_BETTER", [StringType()] * 2)

# Drift -> action: Model Monitor PSI per feature, logged to MODEL_DRIFT_LOG;
# PSI > threshold on any feature runs TASK_TRAIN_CHALLENGER (12h cooldown).
register_proc(
    "check_model_drift", "CHECK_MODEL_DRIFT",
    [StringType(), StringType(), StringType(), FloatType(), IntegerType()],
    execute_as="caller",
)

## 4. Create Task DAG

Creates Snowflake Tasks with dependencies to form a DAG, plus a separate drift-check task that can kick off the DAG:

| Task | Schedule | Depends On | Action |
|------|----------|------------|--------|
| `TASK_TRAIN_CHALLENGER` | Daily 2AM PST | - (root) | Calls `TRAIN_CHALLENGER_MODEL(db, schema, pool)` |
| `TASK_VALIDATE_CHALLENGER` | - | TASK_TRAIN_CHALLENGER | Calls `VALIDATE_CHALLENGER(db, schema)` |
| `TASK_PROMOTE_IF_BETTER` | - | TASK_VALIDATE_CHALLENGER | Calls `PROMOTE_IF_BETTER(db, schema)` |
| `TASK_CHECK_DRIFT` | Daily 2PM PST | - (separate root) | Calls `CHECK_MODEL_DRIFT(...)`; on PSI > 0.2 runs `EXECUTE TASK TASK_TRAIN_CHALLENGER` |

Child tasks automatically run after their parent completes successfully.

In [ ]:
for task in ["TASK_CHECK_DRIFT", "TASK_REFRESH_TRAINING_DATA", "TASK_PROMOTE_IF_BETTER", "TASK_VALIDATE_CHALLENGER", "TASK_TRAIN_CHALLENGER"]:
    try:
        session.sql(f"ALTER TASK IF EXISTS {DATABASE}.{SCHEMA}.{task} SUSPEND").collect()
    except Exception as e:
        print(f"Could not suspend {task}: {e}")
print("Suspended old tasks")

args = f"'{DATABASE}', '{SCHEMA}'"

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER WAREHOUSE={WAREHOUSE} SCHEDULE='USING CRON 0 2 * * * America/Los_Angeles' AS CALL TRAIN_CHALLENGER_MODEL({args}, '{COMPUTE_POOL}')").collect()
print("Created TASK_TRAIN_CHALLENGER (root - daily 2AM)")

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER WAREHOUSE={WAREHOUSE} AFTER {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER AS CALL VALIDATE_CHALLENGER({args})").collect()
print("Created TASK_VALIDATE_CHALLENGER")

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_PROMOTE_IF_BETTER WAREHOUSE={WAREHOUSE} AFTER {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER AS CALL PROMOTE_IF_BETTER({args})").collect()
print("Created TASK_PROMOTE_IF_BETTER")

session.sql(f"CREATE OR REPLACE TASK {DATABASE}.{SCHEMA}.TASK_CHECK_DRIFT WAREHOUSE={WAREHOUSE} SCHEDULE='USING CRON 0 14 * * * America/Los_Angeles' AS CALL CHECK_MODEL_DRIFT({args}, '{MONITOR_NAME}', {PSI_DRIFT_THRESHOLD}, {DRIFT_LOOKBACK_DAYS})").collect()
print("Created TASK_CHECK_DRIFT (daily 2PM, triggers TASK_TRAIN_CHALLENGER on drift)")

## 5. Test Pipeline

Manually execute the pipeline to verify it works:
1. Resume all tasks (required before execution)
2. Execute root task which triggers the full DAG
3. Monitor task completion
4. Review pipeline log to verify champion/challenger logic worked correctly

In [ ]:
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_PROMOTE_IF_BETTER RESUME").collect()
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_VALIDATE_CHALLENGER RESUME").collect()
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER RESUME").collect()
session.sql(f"ALTER TASK {DATABASE}.{SCHEMA}.TASK_CHECK_DRIFT RESUME").collect()
print("Tasks resumed")

In [ ]:
import time
print("Executing pipeline (ML Job takes 5-10 min)...")
session.sql(f"EXECUTE TASK {DATABASE}.{SCHEMA}.TASK_TRAIN_CHALLENGER").collect()

completed = set()
for i in range(180):
    hist = session.sql(f"SELECT NAME, STATE, DATEDIFF('second', SCHEDULED_TIME, COALESCE(COMPLETED_TIME, CURRENT_TIMESTAMP())) as SEC FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START=>DATEADD('minute',-15,CURRENT_TIMESTAMP()))) WHERE DATABASE_NAME='{DATABASE}' AND SCHEMA_NAME='{SCHEMA}' ORDER BY SCHEDULED_TIME DESC").collect()
    for r in hist:
        if r['NAME'] not in completed:
            if r['STATE'] == 'SUCCEEDED':
                print(f"  ✓ {r['NAME']}: {r['SEC']}s")
                completed.add(r['NAME'])
            elif r['STATE'] == 'FAILED':
                print(f"  ✗ {r['NAME']}: FAILED")
                completed.add(r['NAME'])
    if len(completed) >= 3: break
    time.sleep(5)

print("\n📋 Pipeline Log:")
log = session.sql(f"SELECT TO_CHAR(RUN_TIMESTAMP,'HH24:MI:SS') as T, STEP_NAME, STATUS, CHALLENGER_VERSION, ROUND(CHALLENGER_AUC_PR,4) as AUC_PR, ROUND(CHALLENGER_RECALL_AT_1PCT_FPR,4) as RECALL_1PCT_FPR, COMPARISON_BASIS, SUBSTR(MESSAGE,1,80) as MSG FROM PIPELINE_RUN_LOG WHERE RUN_TIMESTAMP > DATEADD('minute',-15,CURRENT_TIMESTAMP()) ORDER BY RUN_TIMESTAMP DESC").to_pandas()
print(log.to_string(index=False))

In [ ]:
print("\n" + "="*60)
print("✅ ML PIPELINE DAG SETUP COMPLETE")
print("="*60)
print("\nPipeline Summary:")
print("  • 3 DAG tasks + drift-triggered TASK_CHECK_DRIFT")
print("  • Runs daily at 2AM PST (or manually via EXECUTE TASK)")
print("  • Champion/Challenger gate: AUC-PR or Recall@1%FPR +1%, neither -1%")
print("\nTo monitor in Snowsight:")
print("  Data > Databases > DEMO_FRAUD_DETECTION_DB > ML_PIPELINE > Tasks")
print("\nTo suspend pipeline:")
print("  ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_TRAIN_CHALLENGER SUSPEND;")
print("  ALTER TASK DEMO_FRAUD_DETECTION_DB.ML_PIPELINE.TASK_CHECK_DRIFT SUSPEND;")